# Skill Retention Regression

**Dataset:** Impact of AI on Students  
**Target:** `Skill_Retention_Score`  
**Task:** Regression

## Problem statement

Estimate skill-retention score from pre-semester, AI-usage, study, anxiety, and institutional features without using other post-semester outcomes.

This notebook is standalone: it contains its own loading, EDA, preprocessing, modelling, validation, interpretation, clustering, limitations, and recommendations. It writes no result files.

## Evidence boundary

The Kaggle source does not document collection or real-versus-synthetic provenance. Results are predictive associations within this file, not causal evidence about student outcomes.

## Aim and objectives

1. Build leakage-controlled linear and nonlinear regressors.
2. Compare at least three real models with a mean baseline.
3. quantify cross-validation and test uncertainty.
4. inspect errors across major, study year, and policy groups.
5. add predictor-only clustering for unsupervised analysis.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
RUN_BALANCED_BACKUP = True
CV_FOLDS = 3 if RUN_BALANCED_BACKUP else 5
BOOTSTRAP_ITERATIONS = 200 if RUN_BALANCED_BACKUP else 1000

def find_dataset():
    relative = Path("Datasets/Impact of AI on Students/ai_student_impact_dataset.csv")
    for start in [Path.cwd(), *Path.cwd().parents]:
        candidate = start / relative
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"Could not locate {relative}")

DATA_PATH = find_dataset()
df = pd.read_csv(DATA_PATH)
df["GPA_Change"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]
df["GPA_Declined"] = (df["GPA_Change"] < 0).astype(int)
print(f"Loaded {len(df):,} rows. CV folds: {CV_FOLDS}. Bootstrap iterations: {BOOTSTRAP_ITERATIONS}.")

In [ ]:
IDENTIFIER = "Student_ID"
OUTCOMES = ["Post_Semester_GPA", "Skill_Retention_Score", "Burnout_Risk_Level"]
EARLY_RISK_FEATURES = [
    "Major_Category", "Year_of_Study", "Pre_Semester_GPA",
    "Weekly_GenAI_Hours", "Primary_Use_Case",
    "Prompt_Engineering_Skill", "Tool_Diversity", "Paid_Subscription",
    "Traditional_Study_Hours", "Perceived_AI_Dependency",
    "Institutional_Policy",
]
EXPANDED_FEATURES = EARLY_RISK_FEATURES + ["Anxiety_Level_During_Exams"]

df["GPA_Change"] = df["Post_Semester_GPA"] - df["Pre_Semester_GPA"]
df["GPA_Declined"] = (df["GPA_Change"] < 0).astype(int)

assert IDENTIFIER not in EARLY_RISK_FEATURES
assert not set(OUTCOMES).intersection(EARLY_RISK_FEATURES)
print("Leakage policy ready. Primary burnout model excludes anxiety and all post-semester outcomes.")

## Complete data audit and EDA

In [ ]:
audit = pd.Series({
    "rows": len(df),
    "columns_original": 16,
    "missing_cells": int(df.iloc[:, :16].isna().sum().sum()),
    "duplicate_rows": int(df.iloc[:, :16].duplicated().sum()),
    "unique_student_ids": int(df["Student_ID"].nunique()),
})
display(audit.to_frame("value"))
display(df.describe(include="all").T)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.countplot(data=df, x="Burnout_Risk_Level", order=["Low", "Medium", "High"], ax=axes[0])
sns.histplot(data=df, x="Skill_Retention_Score", bins=30, kde=True, ax=axes[1])
sns.histplot(data=df, x="GPA_Change", bins=30, kde=True, ax=axes[2])
fig.tight_layout()
plt.show()

## Target-specific EDA

In [ ]:
display(df["Skill_Retention_Score"].describe().to_frame())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(data=df, x="Skill_Retention_Score", bins=35, kde=True, ax=axes[0])
sns.scatterplot(
    data=df.sample(4000, random_state=RANDOM_STATE),
    x="Weekly_GenAI_Hours", y="Skill_Retention_Score",
    hue="Prompt_Engineering_Skill", alpha=0.4, ax=axes[1]
)
fig.tight_layout()
plt.show()

## Preprocessing and leakage contract

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

def make_preprocessor(frame, scale_numeric=True):
    categorical = [
        column for column in frame.columns
        if pd.api.types.is_string_dtype(frame[column])
        or pd.api.types.is_bool_dtype(frame[column])
    ]
    numeric = [column for column in frame.columns if column not in categorical]
    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler()))
    return ColumnTransformer(
        transformers=[
            ("numeric", Pipeline(numeric_steps), numeric),
            (
                "categorical",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                ]),
                categorical,
            ),
        ]
    )

In [ ]:
forbidden = {"Student_ID", "Post_Semester_GPA", "Skill_Retention_Score", "Burnout_Risk_Level"}
assert not forbidden.intersection(EXPANDED_FEATURES)
print("Predictors:", EXPANDED_FEATURES)

## Model comparison with cross-validation and untouched test set

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def regression_metrics(y_true, prediction):
    return {
        "MAE": mean_absolute_error(y_true, prediction),
        "RMSE": mean_squared_error(y_true, prediction) ** 0.5,
        "R2": r2_score(y_true, prediction),
    }

In [ ]:
from sklearn.model_selection import train_test_split, KFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

X = df[EXPANDED_FEATURES]
y = df["Skill_Retention_Score"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)
cv = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
models = {
    "Dummy": DummyRegressor(strategy="mean"),
    "Ridge": Ridge(alpha=1.0),
    "Elastic Net": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=5000),
    "Random forest": RandomForestRegressor(
        n_estimators=120 if RUN_BALANCED_BACKUP else 250,
        min_samples_leaf=3, max_features=0.8,
        n_jobs=-1, random_state=RANDOM_STATE
    ),
    "Histogram gradient boosting": HistGradientBoostingRegressor(
        max_iter=120 if RUN_BALANCED_BACKUP else 220,
        learning_rate=0.07, max_leaf_nodes=31, random_state=RANDOM_STATE
    ),
}
rows = []
fitted_models = {}
for name, estimator in models.items():
    pipe = Pipeline([
        ("preprocess", make_preprocessor(X_train)),
        ("model", estimator),
    ])
    scores = cross_validate(
        pipe, X_train, y_train, cv=cv,
        scoring={"MAE": "neg_mean_absolute_error", "R2": "r2"},
        n_jobs=-1,
    )
    pipe.fit(X_train, y_train)
    prediction = pipe.predict(X_test)
    rows.append({
        "model": name,
        "cv_MAE_mean": -scores["test_MAE"].mean(),
        "cv_R2_mean": scores["test_R2"].mean(),
        "cv_R2_std": scores["test_R2"].std(ddof=1),
        **regression_metrics(y_test, prediction),
    })
    fitted_models[name] = pipe
results = pd.DataFrame(rows).sort_values("RMSE")
display(results)
best_name = results.iloc[0]["model"]
best_model = fitted_models[best_name]
best_prediction = best_model.predict(X_test)
print("Selected model:", best_name)

## Hyperparameter tuning

Tune one competitive model inside cross-validation, then evaluate the refitted configuration once on the untouched test set.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

tuning_pipeline = Pipeline([
    ("preprocess", make_preprocessor(X_train)),
    ("model", HistGradientBoostingRegressor(random_state=RANDOM_STATE)),
])
tuning_space = {
    "model__learning_rate": [0.03, 0.06, 0.10],
    "model__max_leaf_nodes": [15, 31, 63],
    "model__min_samples_leaf": [10, 20, 40],
    "model__l2_regularization": [0.0, 0.1, 1.0],
    "model__max_iter": [100, 180, 260],
}
tuning_search = RandomizedSearchCV(
    tuning_pipeline,
    param_distributions=tuning_space,
    n_iter=4 if RUN_BALANCED_BACKUP else 10,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=1,
    random_state=RANDOM_STATE,
    refit=True,
)
tuning_search.fit(X_train, y_train)
tuned_prediction = tuning_search.predict(X_test)
print("Best tuning parameters:", tuning_search.best_params_)
print("Best cross-validation RMSE:", -tuning_search.best_score_)
display(pd.DataFrame([{
    "model": "Tuned histogram gradient boosting",
    **regression_metrics(y_test, tuned_prediction),
}]))

## Bootstrap uncertainty

In [ ]:
rng = np.random.default_rng(RANDOM_STATE)
positions = np.arange(len(y_test))
bootstrap_rmse = []
bootstrap_r2 = []
for _ in range(BOOTSTRAP_ITERATIONS):
    selected = rng.choice(positions, size=len(positions), replace=True)
    actual = np.asarray(y_test)[selected]
    predicted = np.asarray(best_prediction)[selected]
    bootstrap_rmse.append(mean_squared_error(actual, predicted) ** 0.5)
    bootstrap_r2.append(r2_score(actual, predicted))
print("RMSE bootstrap 95% CI:", np.quantile(bootstrap_rmse, [0.025, 0.975]))
print("R2 bootstrap 95% CI:", np.quantile(bootstrap_r2, [0.025, 0.975]))

## Residual diagnostics

In [ ]:
residuals = np.asarray(y_test) - np.asarray(best_prediction)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(x=best_prediction, y=residuals, alpha=0.25, ax=axes[0])
axes[0].axhline(0, color="black", linestyle="--")
axes[0].set(xlabel="Predicted", ylabel="Residual")
sns.histplot(residuals, bins=35, kde=True, ax=axes[1])
fig.tight_layout()
plt.show()

## Subgroup validation

In [ ]:
test_results = X_test.copy()
test_results["actual"] = np.asarray(y_test)
test_results["predicted"] = best_prediction
subgroup_rows = []
for column in ["Major_Category", "Year_of_Study", "Institutional_Policy"]:
    for value, group in test_results.groupby(column, observed=True):
        subgroup_rows.append({
            "subgroup_field": column,
            "subgroup": value,
            "rows": len(group),
            "MAE": mean_absolute_error(group["actual"], group["predicted"]),
            "RMSE": mean_squared_error(group["actual"], group["predicted"]) ** 0.5,
            "R2": r2_score(group["actual"], group["predicted"]),
        })
display(pd.DataFrame(subgroup_rows))

## Permutation importance

In [ ]:
from sklearn.inspection import permutation_importance

importance_sample = X_test.sample(n=min(3000, len(X_test)), random_state=RANDOM_STATE)
importance_target = y_test.loc[importance_sample.index]
importance = permutation_importance(
    best_model, importance_sample, importance_target,
    scoring="neg_root_mean_squared_error", n_repeats=5,
    random_state=RANDOM_STATE, n_jobs=-1
)
display(pd.DataFrame({
    "feature": importance_sample.columns,
    "importance_mean": importance.importances_mean,
    "importance_std": importance.importances_std,
}).sort_values("importance_mean", ascending=False))

## Predictor-only unsupervised student profiles

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

cluster_features = [
    "Pre_Semester_GPA", "Weekly_GenAI_Hours", "Tool_Diversity",
    "Traditional_Study_Hours", "Perceived_AI_Dependency",
    "Anxiety_Level_During_Exams",
]
cluster_sample = df.sample(
    n=min(10000 if RUN_BALANCED_BACKUP else 20000, len(df)),
    random_state=RANDOM_STATE,
).copy()
cluster_scaled = StandardScaler().fit_transform(cluster_sample[cluster_features])
cluster_scores = []
for k in range(2, 7):
    candidate = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels = candidate.fit_predict(cluster_scaled)
    cluster_scores.append({
        "k": k,
        "silhouette": silhouette_score(
            cluster_scaled, labels, sample_size=min(4000, len(cluster_sample)),
            random_state=RANDOM_STATE,
        ),
    })
cluster_scores = pd.DataFrame(cluster_scores)
selected_k = int(cluster_scores.loc[cluster_scores["silhouette"].idxmax(), "k"])
cluster_model = KMeans(n_clusters=selected_k, n_init=20, random_state=RANDOM_STATE)
cluster_sample["Cluster"] = cluster_model.fit_predict(cluster_scaled)
display(cluster_scores)
display(cluster_sample.groupby("Cluster", observed=True)[
    cluster_features + ["GPA_Change", "Skill_Retention_Score"]
].mean().round(2))
display(pd.crosstab(
    cluster_sample["Cluster"], cluster_sample["Burnout_Risk_Level"], normalize="index"
).round(3))

## Optional GPU extension

In [ ]:
# CUDA neural-network comparison. Scikit-learn baselines above remain CPU models.
RUN_GPU_NEURAL_MODEL = True
GPU_EPOCHS = 40
GPU_BATCH_SIZE = 1024

if RUN_GPU_NEURAL_MODEL:
    import time
    import torch
    from torch import nn
    from torch.utils.data import DataLoader, TensorDataset
    from sklearn.model_selection import train_test_split

    if not torch.cuda.is_available():
        raise RuntimeError(
            "CUDA is unavailable in this kernel. Run 00_gpu_runtime_diagnostics.ipynb "
            "and select the workspace .venv kernel."
        )

    device = torch.device("cuda:0")
    torch.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.benchmark = True

    gpu_preprocessor = make_preprocessor(X_train)
    X_train_array = np.asarray(
        gpu_preprocessor.fit_transform(X_train), dtype=np.float32
    )
    X_test_array = np.asarray(
        gpu_preprocessor.transform(X_test), dtype=np.float32
    )
    y_train_array = np.asarray(y_train, dtype=np.float32)
    target_mean = float(y_train_array.mean())
    target_scale = float(y_train_array.std())
    if target_scale == 0:
        target_scale = 1.0
    y_train_scaled = (y_train_array - target_mean) / target_scale

    train_indices, validation_indices = train_test_split(
        np.arange(len(X_train_array)),
        test_size=0.15,
        random_state=RANDOM_STATE,
    )
    train_dataset = TensorDataset(
        torch.from_numpy(X_train_array[train_indices]),
        torch.from_numpy(y_train_scaled[train_indices, None]),
    )
    validation_features = torch.from_numpy(
        X_train_array[validation_indices]
    ).to(device)
    validation_targets = torch.from_numpy(
        y_train_scaled[validation_indices, None]
    ).to(device)
    train_loader = DataLoader(
        train_dataset,
        batch_size=GPU_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )

    gpu_model = nn.Sequential(
        nn.Linear(X_train_array.shape[1], 128),
        nn.ReLU(),
        nn.BatchNorm1d(128),
        nn.Dropout(0.20),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Dropout(0.10),
        nn.Linear(64, 1),
    ).to(device)
    assert next(gpu_model.parameters()).is_cuda

    loss_function = nn.MSELoss()
    optimizer = torch.optim.AdamW(
        gpu_model.parameters(), lr=1e-3, weight_decay=1e-4
    )
    best_validation_loss = float("inf")
    best_state = None
    history = []
    torch.cuda.reset_peak_memory_stats(device)
    started = time.perf_counter()

    for epoch in range(1, GPU_EPOCHS + 1):
        gpu_model.train()
        training_loss = 0.0
        for feature_batch, target_batch in train_loader:
            feature_batch = feature_batch.to(device, non_blocking=True)
            target_batch = target_batch.to(device, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            prediction_batch = gpu_model(feature_batch)
            loss = loss_function(prediction_batch, target_batch)
            loss.backward()
            optimizer.step()
            training_loss += loss.item() * len(feature_batch)

        gpu_model.eval()
        with torch.inference_mode():
            validation_prediction = gpu_model(validation_features)
            validation_loss = loss_function(
                validation_prediction, validation_targets
            ).item()
        mean_training_loss = training_loss / len(train_dataset)
        history.append((epoch, mean_training_loss, validation_loss))
        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_state = {
                name: value.detach().cpu().clone()
                for name, value in gpu_model.state_dict().items()
            }
        if epoch == 1 or epoch % 5 == 0:
            print(
                f"Epoch {epoch:02d}/{GPU_EPOCHS} | "
                f"train loss={mean_training_loss:.4f} | "
                f"validation loss={validation_loss:.4f}"
            )

    gpu_model.load_state_dict(best_state)
    gpu_model.eval()
    with torch.inference_mode():
        test_features = torch.from_numpy(X_test_array).to(device)
        gpu_prediction_scaled = gpu_model(test_features).squeeze(1)
        gpu_prediction = (
            gpu_prediction_scaled.cpu().numpy() * target_scale + target_mean
        )
    torch.cuda.synchronize(device)
    elapsed_seconds = time.perf_counter() - started

    display(pd.DataFrame([{
        "model": "PyTorch CUDA MLP",
        **regression_metrics(y_test, gpu_prediction),
        "device": str(device),
        "training_seconds": elapsed_seconds,
        "peak_cuda_memory_mb": (
            torch.cuda.max_memory_allocated(device) / 1024**2
        ),
    }]))
    print("CUDA device:", torch.cuda.get_device_name(device))
else:
    print("CUDA neural model skipped.")


## Critical analysis, recommendations, and limitations

- Prefer the model with stable cross-validation and test errors, not the highest training score.
- Treat large residuals and subgroup error differences as priorities for future data collection.
- Perfect cleanliness does not replace provenance, measurement-validity, or external-validation evidence.
- The records have no documented chronology, sampling frame, institution, country, or confirmed real/synthetic status.
- Recommendations are hypotheses for educational investigation, not proof that AI behaviour caused the target.

## Conclusion

This variant tests whether a continuous student outcome is predictable beyond a mean baseline and documents both model uncertainty and the limits of the source data.